# 01 - Exploration des donnees (EDA)

Objectif: comprendre la structure des tables, mesurer la qualite des donnees, et formuler les decisions de nettoyage.

## Points explores
- Volumetrie et schema des trois tables
- Cohesion de l identifiant num_acc
- Doublons et valeurs manquantes
- Anomalies de vma et premiers indices de nettoyage

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from IPython.display import display

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')


In [ ]:
csv_params = {'sep': ',', 'encoding': 'latin1', 'low_memory': False}

df_caract = pd.read_csv('../data/caracteristiques.csv', **csv_params)
df_lieux = pd.read_csv('../data/lieux.csv', **csv_params)
df_usagers = pd.read_csv('../data/usagers.csv', **csv_params)

dfs = {
    'Caract': df_caract,
    'Lieux': df_lieux,
    'Usagers': df_usagers,
}


In [ ]:
for name, df in dfs.items():
    print(f"
{name} -> {df.shape[0]:,} lignes, {df.shape[1]} colonnes")
    display(df.head(3))


In [ ]:
# Harmonisation de num_acc pour analyses croisees
for name, df in dfs.items():
    df['num_acc'] = pd.to_numeric(df['num_acc'], errors='coerce')
    df.dropna(subset=['num_acc'], inplace=True)
    df['num_acc'] = df['num_acc'].astype('int64')
    print(f"{name} -> num_acc dtype: {df['num_acc'].dtype}")


In [ ]:
ids_caract = set(df_caract['num_acc'])

for name, df in dfs.items():
    print(f"
{name}: {df['num_acc'].nunique():,} accidents uniques")
    if name != 'Caract':
        ids = set(df['num_acc'])
        print(f"  -> communs avec Caract: {len(ids_caract & ids):,}")


In [ ]:
for name, df in dfs.items():
    print(f"{name}: {df.duplicated().sum():,} doublons complets")


In [ ]:
def missing_summary(df):
    df_num = df.select_dtypes(include=[np.number]).copy()
    df_num = df_num.replace({-1: np.nan})
    return (df_num.isna().mean() * 100).sort_values(ascending=False)

for name, df in dfs.items():
    print(f"
{name} - top manquants (numeric)")
    display(missing_summary(df).head(10))


In [ ]:
# Vue rapide des manquants (echantillon pour limiter le cout)
sample_lieux = df_lieux.sample(50_000, random_state=42)
msno.matrix(sample_lieux)
plt.show()


In [ ]:
# VMA: distribution et anomalies evidentes
print(df_lieux['vma'].value_counts().head(10))
print(df_lieux['vma'].describe())

print('vma > 130:', int((df_lieux['vma'] > 130).sum()))
print('vma <= 10:', int((df_lieux['vma'] <= 10).sum()))


In [ ]:
# Quelques distributions utiles
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_caract['lum'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Luminosite')
df_caract['atm'].value_counts().sort_index().plot(kind='bar', ax=axes[1], title='Conditions atm')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation (ex: Lieux)
num_cols = df_lieux.select_dtypes(include=[np.number]).columns
corr = df_lieux[num_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation - Lieux (variables numeriques)')
plt.show()
